In [0]:
%run "/Workspace/capstone_project/capstone_project_cyntexa/capstone_bundle/src/silver/customer_scd_type1"

In [0]:
dbutils.widgets.text("catalog", "dev")
catalog = dbutils.widgets.get("catalog")

In [0]:
# ==========================================
# PRODUCT SCD TYPE 2
# ==========================================

# Read cleaned products
products = spark.table(
    f"{catalog}.silver.products_clean"
)

# Create source temporary view
products.createOrReplaceTempView(
    "product_source"
)

In [0]:
%sql
-- Create SCD2 table for the first time

CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog).silver.products_scd2 AS

SELECT
    product_id,
    product_name,
    category,
    price,
    supplier_id,
    current_date() AS effective_date,
    CAST(NULL AS DATE) AS end_date,
    true AS is_current,
    1 AS version

FROM IDENTIFIER(:catalog).silver.products_clean;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Create incoming changed data

CREATE OR REPLACE TEMP VIEW product_source_updated AS

SELECT
    product_id,
    product_name,
    category,

    CASE
        WHEN product_id = 1 THEN 59999.0
        WHEN product_id = 2 THEN 79999.0
        ELSE price
    END AS price,

    supplier_id

FROM product_source; --product_source -> silver_cleaned table 

In [0]:
%sql
SELECT * FROM product_source_updated where product_id IN (1 , 2)

product_id,product_name,category,price,supplier_id
2,Forward Cause,Electronics,79999.0,82
1,Eat Including,Clothing,59999.0,40


In [0]:
%sql
-- Close old versions

MERGE INTO IDENTIFIER(:catalog).silver.products_scd2 AS target

USING product_source_updated AS source

ON target.product_id = source.product_id
AND target.is_current = true

WHEN MATCHED AND (
       target.product_name <> source.product_name
    OR target.category <> source.category
    OR target.price <> source.price
    OR target.supplier_id <> source.supplier_id
)

THEN UPDATE SET
    target.end_date = current_date(),
    target.is_current = false;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,2,0,0


In [0]:
%sql
-- Prepare new versions

CREATE OR REPLACE TEMP VIEW product_new_versions AS

SELECT
    source.product_id,
    source.product_name,
    source.category,
    source.price,
    source.supplier_id,

    current_date() AS effective_date,

    CAST(NULL AS DATE) AS end_date,

    true AS is_current,

    COALESCE(MAX(target.version), 0) + 1 AS version

FROM product_source_updated AS source

LEFT JOIN IDENTIFIER(:catalog).silver.products_scd2 AS target
    ON source.product_id = target.product_id

GROUP BY
    source.product_id,
    source.product_name,
    source.category,
    source.price,
    source.supplier_id;

In [0]:
%sql
-- Insert new versions

MERGE INTO IDENTIFIER(:catalog).silver.products_scd2 AS target

USING product_new_versions AS source

ON target.product_id = source.product_id
AND target.is_current = true

WHEN NOT MATCHED THEN

INSERT (
    product_id,
    product_name,
    category,
    price,
    supplier_id,
    effective_date,
    end_date,
    is_current,
    version
)

VALUES (
    source.product_id,
    source.product_name,
    source.category,
    source.price,
    source.supplier_id,
    source.effective_date,
    source.end_date,
    source.is_current,
    source.version
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,0,0,2
